# Check database

In [0]:
%sql
SHOW DATABASES;

databaseName
airline_tables
default
information_schema


In [0]:
%sql
-- set the schema
USE SCHEMA airline_tables;

# Create Dimension Tables

## dim_location

In [0]:
%sql
-- create dim_location
CREATE OR REPLACE TABLE airline_tables.dim_location (
    location_key   INT,
    country_code   STRING,
    country_name   STRING,
    continent_code STRING,
    continent_name STRING
);

COPY INTO airline_tables.dim_location
FROM '/Volumes/workspace/airline_tables/airline_fact_dim/dim_location.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true');

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
249,249,0


## dim_passenger

In [0]:
%sql
-- create dim_passenger
CREATE OR REPLACE TABLE airline_tables.dim_passenger (
    passenger_key  INT,
    passenger_id   STRING,
    age            INT,
    age_group      STRING,
    gender         STRING,
    nationality    STRING
);

COPY INTO airline_tables.dim_passenger
FROM '/Volumes/workspace/airline_tables/airline_fact_dim/dim_passenger.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true');

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
97693,97693,0


## dim_airport

In [0]:
%sql
-- dim_airport
CREATE OR REPLACE TABLE airline_tables.dim_airport (
    airport_key    INT,
    airport_code   STRING,
    airport_name   STRING,
    airport_type   STRING,
    municipality   STRING,
    country_key    INT  -- references dim_location(location_key)
);

COPY INTO airline_tables.dim_airport
FROM '/Volumes/workspace/airline_tables/airline_fact_dim/dim_airport_2.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true');

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
9709,9709,0


## dim_date

In [0]:
%sql
-- dim_date
CREATE OR REPLACE TABLE airline_tables.dim_date (
    date_key  INT,
    date      DATE,
    day       INT,
    week      INT,
    month     INT,
    quarter   INT,
    year      INT
);

COPY INTO airline_tables.dim_date
FROM '/Volumes/workspace/airline_tables/airline_fact_dim/dim_date.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true');

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
364,364,0


## dim_flight_status

In [0]:
%sql
-- dim_flight_status
CREATE OR REPLACE TABLE airline_tables.dim_flight_status (
    flight_status_key  INT,
    flight_status      STRING
);

COPY INTO airline_tables.dim_flight_status
FROM '/Volumes/workspace/airline_tables/airline_fact_dim/dim_flight_status.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true');

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
3,3,0


# Create Fact Table

In [0]:
%sql
-- fact table: fact_flight_activity
CREATE OR REPLACE TABLE airline_tables.fact_flight_activity (
    flight_activity_id  INT,
    passenger_key       INT,   -- references dim_passenger(passenger_key)
    airport_key         INT,   -- references dim_airport(airport_key)
    date_key            INT,   -- references dim_date(date_key)
    flight_status_key   INT,   -- references dim_flight_status(flight_status_key)
    passenger_count     INT,
    is_international    INT
);

COPY INTO airline_tables.fact_flight_activity
FROM '/Volumes/workspace/airline_tables/airline_fact_dim/fact_flight_activity_4.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true');

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
97693,97693,0


# Set Primary & Foreign Key Constraints

In [0]:
%sql

-- Make primary key NOT NULL
ALTER TABLE airline_tables.dim_passenger ALTER COLUMN passenger_key SET NOT NULL;
ALTER TABLE airline_tables.dim_airport ALTER COLUMN airport_key SET NOT NULL;
ALTER TABLE airline_tables.dim_date ALTER COLUMN date_key SET NOT NULL;
ALTER TABLE airline_tables.dim_flight_status ALTER COLUMN flight_status_key SET NOT NULL;
ALTER TABLE airline_tables.dim_location ALTER COLUMN location_key SET NOT NULL;
ALTER TABLE airline_tables.fact_flight_activity ALTER COLUMN flight_activity_id SET NOT NULL;

-- Add primary key constraints
ALTER TABLE airline_tables.dim_passenger
ADD CONSTRAINT pk_passenger PRIMARY KEY (passenger_key);

ALTER TABLE airline_tables.dim_airport
ADD CONSTRAINT pk_airport PRIMARY KEY (airport_key);

ALTER TABLE airline_tables.dim_date
ADD CONSTRAINT pk_date PRIMARY KEY (date_key);

ALTER TABLE airline_tables.dim_flight_status
ADD CONSTRAINT pk_flight_status PRIMARY KEY (flight_status_key);

ALTER TABLE airline_tables.dim_location
ADD CONSTRAINT pk_location PRIMARY KEY (location_key);

-- Add primary key to fact table
ALTER TABLE airline_tables.fact_flight_activity
ADD CONSTRAINT pk_fact PRIMARY KEY (flight_activity_id);

In [0]:
%sql
-- Add foreign key constraints to show ERD diagram
ALTER TABLE airline_tables.fact_flight_activity
ADD CONSTRAINT fk_passenger FOREIGN KEY (passenger_key) REFERENCES airline_tables.dim_passenger(passenger_key);

ALTER TABLE airline_tables.fact_flight_activity
ADD CONSTRAINT fk_airport FOREIGN KEY (airport_key) REFERENCES airline_tables.dim_airport(airport_key);

ALTER TABLE airline_tables.fact_flight_activity
ADD CONSTRAINT fk_date FOREIGN KEY (date_key) REFERENCES airline_tables.dim_date(date_key);

ALTER TABLE airline_tables.fact_flight_activity
ADD CONSTRAINT fk_flight_status FOREIGN KEY (flight_status_key) REFERENCES airline_tables.dim_flight_status(flight_status_key);

-- Add foreign key from dim_airport to dim_location
ALTER TABLE airline_tables.dim_airport
ADD CONSTRAINT fk_country FOREIGN KEY (country_key) REFERENCES airline_tables.dim_location(location_key);